# M4 · ChromaDB 검색 기준선
문서의 글자 패턴을 숫자로 바꿔 ChromaDB에 넣는다. 키 없이 검색 흐름을 먼저 확인하려고 문자 n-gram 벡터를 쓴다. 이 벡터는 의미 임베딩과 다르다.

In [ ]:
from pathlib import Path
import hashlib,json
import numpy as np
import chromadb
from sklearn.feature_extraction.text import HashingVectorizer
from IPython.display import display
ROOT=next((p for p in [Path.cwd(),*Path.cwd().parents] if (p/'pyproject.toml').exists()),None)
assert ROOT is not None,'SANUP-P 프로젝트 폴더에서 실행하세요.'
PERSONAL=ROOT/'data'/'personal'
CORPUS=PERSONAL/'corpus';DB=ROOT/'chroma_db'/'personal';DB.mkdir(parents=True,exist_ok=True)
vectorizer=HashingVectorizer(analyzer='char',ngram_range=(2,4),n_features=1024,
                             alternate_sign=False,norm='l2',dtype=np.float32)
def vectors(texts):return vectorizer.transform(texts).toarray().astype(np.float32)

## 청크 읽고 원본 버전 확인

In [ ]:
chunk_path=CORPUS/'chunks.jsonl'
assert chunk_path.exists(),'08 청킹 노트북을 먼저 실행하세요.'
raw=chunk_path.read_bytes()
corpus_sha=hashlib.sha256(raw).hexdigest()
chunks=[json.loads(line) for line in raw.decode('utf-8').splitlines() if line.strip()]
assert len(chunks)>6000 and len({x['chunk_id'] for x in chunks})==len(chunks)
collection_name='sanup_lex_'+corpus_sha[:12]
print('청크',len(chunks),'색인',collection_name)

## 배치로 넣고 건수 확인

In [ ]:
client=chromadb.PersistentClient(path=str(DB))
collection=client.get_or_create_collection(collection_name,metadata={'hnsw:space':'cosine'},embedding_function=None)
if collection.count()!=len(chunks):
    for start in range(0,len(chunks),200):
        batch=chunks[start:start+200]
        texts=[x['text'] for x in batch]
        metadata=[{'doc_id':x['doc_id'],'source_id':x['source_id'],
                   'source_type':x['source_type'],'title':x['title'],
                   'organization':x['organization'],'source_url':x['source_url'],
                   'page':int(x['page'] or 0),'industry_major':x['industry_major'] or '',
                   'equipment':x['equipment'] or '',
                   'extraction_method':x.get('extraction_method','pdf_text'),
                   'ocr_review_required':bool(x.get('ocr_review_required',False))} for x in batch]
        collection.upsert(ids=[x['chunk_id'] for x in batch],
                          embeddings=vectors(texts).tolist(),documents=texts,metadatas=metadata)
assert collection.count()==len(chunks)
index_meta={'collection':collection_name,'corpus_sha256':corpus_sha,'chunks':len(chunks),
            'embedding_method':'HashingVectorizer char 2-4 grams 1024 dims; lexical baseline',
            'semantic_embedding_verified':False}
(CORPUS/'index_meta.json').write_text(json.dumps(index_meta,ensure_ascii=False,indent=2),encoding='utf-8')
print('색인된 청크',collection.count())

## 같은 문서의 청크는 한 번만 세고 검색

In [ ]:
def search(query,k=5,source_type=None):
    where={'source_type':source_type} if source_type else None
    found=collection.query(query_embeddings=vectors([query]).tolist(),
                           n_results=min(max(k*8,20),collection.count()),
                           where=where,include=['documents','metadatas','distances'])
    results=[];seen=set()
    for text,meta,distance in zip(found['documents'][0],found['metadatas'][0],found['distances'][0]):
        if meta['doc_id'] in seen:continue
        seen.add(meta['doc_id'])
        results.append({'doc_id':meta['doc_id'],'title':meta['title'],
                        'source_type':meta['source_type'],'source_url':meta['source_url'],
                        'page':meta['page'],'similarity':round(1-float(distance),4),
                        'excerpt':text[:180]})
        if len(results)==k:break
    return results
for query in ['이동식 사다리 안전한 사용 방법','지게차 하역작업 사고 원인']:
    print('질문:',query)
    display(search(query,5))

## 색인 결과와 검색 한계

- 6,845개 청크를 문자 n-gram 기반 로컬 Chroma 색인에 넣었다. 이 색인은 의미 임베딩이 아니라 어휘 겹침을 보는 로컬 기준선이며 API 비용이 들지 않는다.
- 색인 메타데이터의 청크 수와 코퍼스 SHA-256을 보관한다. 검색어 표현이 원문과 다르면 관련 의미가 있어도 놓칠 수 있고, 같은 단어가 우연히 겹치는 청크가 올라올 수 있다.
- OCR로 만든 청크에는 PDF 문서 ID·페이지와 `ocr_review_required` 표시를 연결했다. 검색 결과만 읽고 인용을 끝내지 말고 PDF 쪽을 확인한다.

### 재실행·운영 주의

- 코퍼스가 바뀌었으면 새 해시로 색인을 만들고 문서 수만 맞는지 보지 말고 색인 metadata의 코퍼스 해시도 함께 대조한다.
- 이 문자 검색 기준선은 M6 BM25와 같지 않다. 검색 방식과 검색 대상·질문 집합이 다르면 점수를 직접 비교하지 않는다.
- 벡터 DB는 개인 작업 데이터로만 관리하며 Git에 올리지 않는다.
